In [1]:
"""
2D Poisson with Gaussian source:
    -Δu = f(x,y; x0, y0, nu)  on [0,1]^2,  u = 0 on boundary.

TRAINING STAGE ONLY
-------------------
This file only trains the meta-conditioned KAPI RBF PINN once and saves the
trained weights. No post-processing plots are generated here.
"""

import os
import random
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ------------------------------------------------------------
# Output directories
# ------------------------------------------------------------
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

DATA_DIR = "TC_01_FIGURES"
os.makedirs(DATA_DIR, exist_ok=True)

MODEL_PATH = "TC_01_FIGURES/meta_poisson_model.pt"
HISTORY_PATH = "TC_01_FIGURES/meta_poisson_training_history.npz"

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print("Random seed set to:", SEED)

# ------------------------------------------------------------
# 2D Gaussian source term for Poisson
# f(x,y) = (1/(2πν^2)) exp(-r^2/(2ν^2))
# ------------------------------------------------------------
def gaussian_source(x, y, x0, y0, nu):
    r2 = (x - x0) ** 2 + (y - y0) ** 2
    return (1.0 / (2.0 * torch.pi * nu ** 2)) * torch.exp(-r2 / (2.0 * nu ** 2))


# ------------------------------------------------------------
# KAPI-RBF PINN model (meta-conditioned on PDE parameters)
# ------------------------------------------------------------
class KAPIRBF_PINN(nn.Module):
    def __init__(self, M=128, hidden_meta=64):
        super().__init__()
        self.M = M
        self.meta = nn.Sequential(
            nn.Linear(3, hidden_meta),
            nn.Tanh(),
            nn.Linear(hidden_meta, hidden_meta),
            nn.Tanh(),
            nn.Linear(hidden_meta, 4 * M),
        )
        self.c = nn.Parameter(1e-2 * torch.randn(M))

    def _forward_raw(self, p, xy):
        assert p.shape[0] == 1, "This implementation assumes a single p per call."

        p_mean = p.new_tensor([0.5, 0.5, 0.075])
        p_std  = p.new_tensor([0.1, 0.1, 0.025])
        p_norm = (p - p_mean) / p_std

        meta_out = self.meta(p_norm).view(1, 4, self.M)
        g_logits   = meta_out[:, 0, :]
        mu_raw_x   = meta_out[:, 1, :]
        mu_raw_y   = meta_out[:, 2, :]
        log_sigraw = meta_out[:, 3, :]

        g = torch.sigmoid(g_logits).view(self.M)
        mu_x = torch.sigmoid(mu_raw_x).view(self.M)
        mu_y = torch.sigmoid(mu_raw_y).view(self.M)
        sigma = torch.nn.functional.softplus(log_sigraw).view(self.M) + 5e-4

        x = xy[:, 0:1]
        y = xy[:, 1:2]
        x_diff = x - mu_x.view(1, self.M)
        y_diff = y - mu_y.view(1, self.M)
        r2 = (x_diff ** 2 + y_diff ** 2) / (sigma.view(1, self.M) ** 2)
        Phi = torch.exp(-r2)

        coeff = g * self.c
        u_raw = Phi @ coeff
        return u_raw, (g, mu_x, mu_y, sigma)

    def forward(self, p, xy):
        u_raw, aux = self._forward_raw(p, xy)
        x, y = xy[:, 0:1], xy[:, 1:2]
        bc_factor = x * (1.0 - x) * y * (1.0 - y)
        u = u_raw * bc_factor.view(-1)
        return u, aux


# ------------------------------------------------------------
# p-dependent collocation and boundary points on [0,1]^2
# ------------------------------------------------------------
def sample_collocation_points_p_dependent(
    N_int,
    N_bc,
    p,
    alpha=None,
    sigma_factor=3.0,
    device=device
):
    x0, y0, nu = p[0]
    if alpha is None:
        alpha = 0.9 if nu.item() < 0.06 else 0.7

    N_loc = int(alpha * N_int)
    N_uni = N_int - N_loc

    if N_loc > 0:
        loc_x = x0 + sigma_factor * nu * torch.randn(N_loc, 1, device=device)
        loc_y = y0 + sigma_factor * nu * torch.randn(N_loc, 1, device=device)
        loc_x = loc_x.clamp(0.0, 1.0)
        loc_y = loc_y.clamp(0.0, 1.0)
        xy_loc = torch.cat([loc_x, loc_y], dim=1)
    else:
        xy_loc = torch.empty(0, 2, device=device)

    if N_uni > 0:
        uni_x = torch.rand(N_uni, 1, device=device)
        uni_y = torch.rand(N_uni, 1, device=device)
        xy_uni = torch.cat([uni_x, uni_y], dim=1)
    else:
        xy_uni = torch.empty(0, 2, device=device)

    xy_int = torch.cat([xy_loc, xy_uni], dim=0)

    if N_bc > 0:
        N_side = N_bc // 4
        t = torch.rand(N_side, 1, device=device)
        xb = torch.cat([
            torch.zeros(N_side, 1, device=device),
            torch.ones(N_side, 1, device=device),
            t,
            t
        ], dim=0)
        yb = torch.cat([
            t,
            t,
            torch.zeros(N_side, 1, device=device),
            torch.ones(N_side, 1, device=device)
        ], dim=0)
        xy_bc = torch.cat([xb, yb], dim=1)
    else:
        xy_bc = torch.empty(0, 2, device=device)

    return xy_int, xy_bc


# ------------------------------------------------------------
# Poisson residual: -Δu - f(x,y;p)
# ------------------------------------------------------------
def poisson_residual(model, p, xy_int):
    xy_int.requires_grad_(True)
    u, _ = model(p, xy_int)

    grads = torch.autograd.grad(
        u, xy_int,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]
    u_x = grads[:, 0]
    u_y = grads[:, 1]

    grads2_x = torch.autograd.grad(
        u_x, xy_int,
        grad_outputs=torch.ones_like(u_x),
        create_graph=True
    )[0]
    grads2_y = torch.autograd.grad(
        u_y, xy_int,
        grad_outputs=torch.ones_like(u_y),
        create_graph=True
    )[0]

    u_xx = grads2_x[:, 0]
    u_yy = grads2_y[:, 1]
    laplace_u = u_xx + u_yy

    x = xy_int[:, 0]
    y = xy_int[:, 1]
    x0, y0, nu = p[0, 0], p[0, 1], p[0, 2]
    f_val = gaussian_source(x, y, x0, y0, nu)

    return -laplace_u - f_val


# ------------------------------------------------------------
# Sample PDE parameters p = (x0, y0, nu)
# ------------------------------------------------------------
x0_min, x0_max = 0.4, 0.6
y0_min, y0_max = 0.4, 0.6
nu_min, nu_max = 0.05, 0.1

def sample_pde_param(nu_min_curr=None, nu_max_curr=None):
    if nu_min_curr is None:
        nu_min_curr = nu_min
    if nu_max_curr is None:
        nu_max_curr = nu_max

    x0 = x0_min + (x0_max - x0_min) * torch.rand(1)
    y0 = y0_min + (y0_max - y0_min) * torch.rand(1)
    u = torch.rand(1)
    nu_min_t = torch.tensor(nu_min_curr)
    nu_max_t = torch.tensor(nu_max_curr)
    nu = 10.0 ** (torch.log10(nu_min_t) +
                  (torch.log10(nu_max_t) - torch.log10(nu_min_t)) * u)

    p = torch.stack([x0, y0, nu], dim=-1)
    return p.to(device)


# ------------------------------------------------------------
# Meta-training loop
# ------------------------------------------------------------
def train_meta_poisson(
    n_int=2048,
    n_bc=256,
    epochs=2000,
    lr=1e-3,
    M=128,
    tasks_per_batch=4
):
    model = KAPIRBF_PINN(M=M, hidden_meta=64).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = StepLR(optimizer, step_size=1000, gamma=0.5)

    best_loss = float("inf")
    best_state = None
    history = {"epoch": [], "total_loss": [], "pde_loss": [], "bc_loss": [], "lr": []}

    for ep in range(1, epochs + 1):
        optimizer.zero_grad()
        total_loss = 0.0
        total_pde  = 0.0
        total_bc   = 0.0

        if ep <= epochs // 2:
            nu_min_curr, nu_max_curr = 0.08, nu_max
        else:
            nu_min_curr, nu_max_curr = nu_min, nu_max

        for _ in range(tasks_per_batch):
            p = sample_pde_param(nu_min_curr, nu_max_curr)
            xy_int, xy_bc = sample_collocation_points_p_dependent(
                n_int, n_bc, p, alpha=None, sigma_factor=3.0, device=device
            )
            res_int = poisson_residual(model, p, xy_int)
            loss_pde = torch.mean(res_int ** 2)

            if xy_bc.shape[0] > 0:
                u_bc, (g_bc, _, _, _) = model(p, xy_bc)
                loss_bc = torch.mean(u_bc ** 2)
            else:
                dummy_xy = torch.zeros(1, 2, device=device)
                _, (g_bc, _, _, _) = model(p, dummy_xy)
                loss_bc = torch.tensor(0.0, device=device)

            loss_sparse = 1e-5 * torch.mean(torch.abs(g_bc))
            loss_task = loss_pde + loss_sparse
            total_loss += loss_task
            total_pde  += loss_pde
            total_bc   += loss_bc

        total_loss /= tasks_per_batch
        total_pde  /= tasks_per_batch
        total_bc   /= tasks_per_batch

        total_loss.backward()
        optimizer.step()
        scheduler.step()

        current_lr = scheduler.get_last_lr()[0]
        history["epoch"].append(ep)
        history["total_loss"].append(total_loss.item())
        history["pde_loss"].append(total_pde.item())
        history["bc_loss"].append(total_bc.item())
        history["lr"].append(current_lr)

        if total_loss.item() < best_loss:
            best_loss = total_loss.item()
            best_state = model.state_dict()

        if ep % 200 == 0:
            print(
                f"Epoch {ep:5d} | Loss: {total_loss.item():.3e} "
                f"(PDE: {total_pde.item():.3e}, BC(mon.): {total_bc.item():.3e}, lr={current_lr:.1e})"
            )

    print(f"\nBest training loss = {best_loss:.3e}")
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history


# ------------------------------------------------------------
# Main: train once and save model
# ------------------------------------------------------------
if __name__ == "__main__":
    n_int = 2048
    n_bc = 256
    epochs = 2000
    lr = 1e-3
    M = 128
    tasks_per_batch = 4

    model, history = train_meta_poisson(
        n_int=n_int,
        n_bc=n_bc,
        epochs=epochs,
        lr=lr,
        M=M,
        tasks_per_batch=tasks_per_batch
    )

    torch.save(model.state_dict(), MODEL_PATH)
    print(f"\nModel saved to: {MODEL_PATH}")

    np.savez_compressed(
        HISTORY_PATH,
        epoch=np.asarray(history["epoch"]),
        total_loss=np.asarray(history["total_loss"]),
        pde_loss=np.asarray(history["pde_loss"]),
        bc_loss=np.asarray(history["bc_loss"]),
        lr=np.asarray(history["lr"]),
    )
    print(f"Training history saved to: {HISTORY_PATH}")


Using device: cpu
Random seed set to: 1234
Epoch   200 | Loss: 8.349e-02 (PDE: 8.348e-02, BC(mon.): 0.000e+00, lr=1.0e-03)
Epoch   400 | Loss: 2.830e-02 (PDE: 2.830e-02, BC(mon.): 0.000e+00, lr=1.0e-03)
Epoch   600 | Loss: 2.843e-02 (PDE: 2.842e-02, BC(mon.): 0.000e+00, lr=1.0e-03)
Epoch   800 | Loss: 4.900e-02 (PDE: 4.899e-02, BC(mon.): 0.000e+00, lr=1.0e-03)
Epoch  1000 | Loss: 1.882e-02 (PDE: 1.882e-02, BC(mon.): 0.000e+00, lr=5.0e-04)
Epoch  1200 | Loss: 1.212e-01 (PDE: 1.212e-01, BC(mon.): 0.000e+00, lr=5.0e-04)
Epoch  1400 | Loss: 6.843e-02 (PDE: 6.842e-02, BC(mon.): 0.000e+00, lr=5.0e-04)
Epoch  1600 | Loss: 4.818e-02 (PDE: 4.817e-02, BC(mon.): 0.000e+00, lr=5.0e-04)
Epoch  1800 | Loss: 5.720e-02 (PDE: 5.719e-02, BC(mon.): 0.000e+00, lr=5.0e-04)
Epoch  2000 | Loss: 5.148e-02 (PDE: 5.147e-02, BC(mon.): 0.000e+00, lr=2.5e-04)

Best training loss = 1.609e-02

Model saved to: TC_01_FIGURES/meta_poisson_model.pt
Training history saved to: TC_01_FIGURES/meta_poisson_training_history.n